### Langchain入门

In [3]:
import os
from statistics import multimode

# 1.加载环境变量
from dotenv import load_dotenv
from tornado.gen import multi

load_dotenv()
from langchain.tools import tool
from langchain.agents import *
# 2.定义工具，基础版，通过注释描述工具
@tool
def getWeather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"


# 3.定义Agent
agent = create_agent(
    "deepseek-chat", # 模型名称（必须是LangChain支持的模型）
    tools=[getWeather] # 工具集
)

# 4.调用模型
print("🚀 正在调用大模型...")
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "杭州今天天气如何?"}
    ]
})

# 5.打印结果
print(response)

🚀 正在调用大模型...


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

## 初始化模型

In [ ]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [4]:
response = model.invoke([
    {
        "role": "system",
        "content": "你是火箭队的武藏"
    },
    {
        "role": "user",
        "content": "请你自我介绍"
    }
])
print(response.content)

NameError: name 'model' is not defined

In [ ]:
stream = model.stream("你是谁？")
for chunk in stream:
    print(chunk.content,end="",flush=True)

## 创建Agent

In [ ]:
agent = create_agent(model=model)

# response = agent.invoke("你是谁？")
response = agent.stream({
    "messages": [
        {
            "role": "system",
            "content": "你是一个 helpful的助手"
        },
        {
            "role": "user",
            "content": "你是谁？"
        }
    ]
},stream_mode="messages")
for token,metadata in response:
    print(token.content,end="",flush=True)

### 消息类型

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

# 创建Agent
agent = create_agent(model="deepseek-chat")

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})

print(response)

### 多模态消息

In [11]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import os
load_dotenv()
model = init_chat_model(model="qwen3.5-plus-2026-02-15",
                        model_provider="openai",
                        api_key=os.getenv("DASHSCOPE_API_KEY"),
                        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
agent = create_agent(model=model)

multi_message = HumanMessage(content=[
    {
        "type":"image",
        "url":"https://cdn.nlark.com/yuque/0/2026/png/64933360/1776757111973-7a3ed782-9e48-4ea9-8df1-ac449c4a25bd.png?x-oss-process=image%2Fformat%2Cwebp"
    },
    {
        "type":"text",
        "text":"这个图片描述了什么？"
    }
])
#flush是Python中的一个参数，用于控制是否立即输出，还是等待缓冲区满后再输出
for token,metadata in agent.stream(
    {"messages": [multi_message]},
    stream_mode="messages"):
    print(token.content,end="",flush=True)



这张图片是一张**序列图（Sequence Diagram）**，详细描述了一个 **AI Agent（智能体）如何通过调用外部工具（Function Calling / Tool Use）来回答用户关于天气的提问** 的完整技术流程。

以下是图中展示的具体步骤和交互逻辑：

**1. 主要参与者（从左到右）：**
*   **用户 (User)**：发起请求的人。
*   **Agent**：中间协调者，负责接收用户请求、与模型交互以及执行具体工具。
*   **模型 (Model)**：大语言模型（LLM），负责理解意图、决定是否需要调用工具以及生成最终回答。

**2. 详细流程解析：**

1.  **用户提问**：用户向 Agent 发送指令：“杭州天气如何？”
2.  **参数组装**：Agent 接收到问题后，将**用户问题 (message)** 和预设的**工具信息 (tools)**（图中黄色便签显示：工具名为 `get_weather`，描述为查询天气，参数为 `city`）组装成请求参数。
3.  **第一次调用模型**：Agent 将组装好的参数发送给模型。
4.  **模型判断**：模型接收请求，进行内部推理（右侧自循环箭头），判断用户意图是否需要调用工具。模型识别出需要调用 `get_weather` 工具，并提取出参数“杭州”。
5.  **返回工具调用指令**：模型告诉 Agent 需要调用工具（图中虚线箭头“【调用工具】”），并返回了具体的工具名称和参数（黄色便签：`name: get_weather`, `parameter: 杭州`）。
6.  **执行工具**：Agent 根据模型的指令，实际执行了天气查询工具（右侧自循环箭头“执行工具”）。
7.  **第二次调用模型（带结果）**：Agent 获取到工具的执行结果（例如：“The weather in 杭州 is sunny!”），并将这个结果再次发送给模型。
8.  **生成回答**：模型结合工具返回的具体数据，进行分析和润色（右侧自循环箭头“分析结果，生成回答”），生成自然语言的回复。
9.  **返回最终答案**：模型将生成的回答（“杭州天气晴朗！”）返回给 Agent。
10. **回复用户**：Agent 最终将答案“杭州天气晴朗！”发送给用户。

**总结：

### 本地图片需要先转成base64编码格式

In [12]:
from ipywidgets import FileUpload
from IPython.display import display
uploader = FileUpload(accept='*',multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [13]:
print(uploader.value)

({'name': '集装箱.png', 'type': 'image/png', 'size': 2741143, 'content': <memory at 0x116aadc00>, 'last_modified': datetime.datetime(2026, 4, 3, 8, 50, 54, 643000, tzinfo=datetime.timezone.utc)},)


In [15]:
import base64
#获取第一个(也是唯一一个)上传的文件
uploaded_file =uploader.value[0]
# 获取其内存视图
content_mv = uploaded_file["content"]
# 转换内存视图->字节
img_bytes =bytes(content_mv)

In [16]:
import base64

# 例如，有一个用户上传的文件，是字节格式img_bytes，我们先将其进行base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

# 调用Agent，发送消息
response = agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
)
for token,metadata in response:
    print(token.content,end="",flush=True)


KeyboardInterrupt: 

# 提示词工程

添加身份，和特定的指令

In [18]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

school_name = "黑马程序员"

In [19]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

金曦城（Auroria）——悬浮于金星硫酸云层中的浮空都市群，依托碳纤维骨架与抗酸薄膜穹顶，核心能源来自大气层深处的超高温涡轮阵列，城市底部持续释放蓝色等离子光柱穿透云海。

### 结构化输出


In [ ]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response['messages'][-1].content)

### Langchain简化了结构化输出

In [ ]:
from pydantic import BaseModel
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [ ]:
# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

In [ ]:
city = response['structured_response']

# 工具类的定义

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5